# Frequency-security features, frozen cuts and coefficient export

This notebook uses real supplied unit inputs and the already fitted LP35 coefficients. It **does not retrain** a classifier. It verifies every outage-specific Big-M, reconstructs the generated include byte-for-byte, and works through one explicitly constructed teaching dispatch.

For outage $k$, use
$$L=p_k,\quad E=\sum_{i\ne k}H_i S_i x_i,\quad G=\sum_{i\ne k}K_iS_i x_i/f_0,\quad R=\sum_{i\ne k}(P_i^{\max}x_i-p_i).$$
The frozen safe-margin inequality is
$$c_0+c_L L+c_E E+c_G G+c_R R+M_k(1-x_k)\ge0.$$
$E$ is MW s, not seconds; $G$ is MW/Hz. The interrupted machine contributes neither inertia nor response. When $x_k=0$, its loss is exactly zero and Big-M relaxes the otherwise irrelevant margin.

In [1]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd


def find_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'GAMS_LaPalma' / 'START_LAPALMA.gms').is_file():
            return candidate
    raise FileNotFoundError('Extract the complete repository, then open this notebook inside it.')


def read_inc_table(path, row_dimensions):
    """Read the shipped XLS2GMS numerical tables; never evaluate source code.

    Leading star-comments and blank padding are ignored. Dotted row tuples,
    e.g. i1 . b1, identify row dimensions; the first data line is the header.
    The function intentionally supports this repository's tables, not all GAMS.
    """
    lines = [s.strip() for s in Path(path).read_text(encoding='utf-8-sig').splitlines()
             if s.strip() and not s.lstrip().startswith('*')]
    if not lines:
        raise ValueError(f'Empty table: {path}')
    columns = lines[0].split()
    records, labels = [], []
    for line_number, line in enumerate(lines[1:], start=2):
        parts = [x for x in line.split() if x != '.']
        keys, fields = parts[:row_dimensions], parts[row_dimensions:]
        if len(keys) != row_dimensions or len(fields) != len(columns):
            raise ValueError(f'Malformed table row {line_number} in {path}')
        values = [0.0 if v.lower() == 'eps' else float(v) for v in fields]
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite table values in {path}')
        labels.append(keys[0] if row_dimensions == 1 else tuple(keys))
        records.append(values)
    index = pd.Index(labels) if row_dimensions == 1 else pd.MultiIndex.from_tuples(labels)
    if not index.is_unique:
        raise ValueError(f'Duplicate row identifiers in {path}')
    return pd.DataFrame(records, index=index, columns=columns)


def load_inputs(root):
    inc = Path(root) / 'GAMS_LaPalma' / 'inc'
    ids = [f'i{i}' for i in range(1, 12)]
    def vector(name, col):
        frame = read_inc_table(inc / name, 1)
        return frame.loc[ids, col].to_numpy(dtype=float)
    blocks = read_inc_table(inc / 'block_max.inc', 1).loc[ids, ['b1','b2','b3']]
    units = pd.DataFrame({'unit': ids, 'pmax_mw': blocks.sum(axis=1).to_numpy(),
        'pmin_mw': vector('aux9.inc','tr4'), 'inertia_s': vector('auxinertia.inc','column1'),
        'mbase': vector('auxmbase.inc','column1'), 'gain_pu': vector('auxk.inc','column1'),
        'no_load_keur_h': vector('aux4.inc','tr4'), 'ramp_up_mw_h': vector('aux5.inc','tr4'),
        'ramp_down_mw_h': vector('aux6.inc','tr4'), 'min_up_h': vector('aux8.inc','tr4'),
        'min_down_h': vector('aux7.inc','tr4'), 'initial_on_age_h': vector('aux3.inc','column1'),
        'initial_off_age_h': vector('aux2.inc','column1'), 'initial_power_mw': vector('aux10.inc','column1')})
    units['initial_commitment'] = (units.initial_on_age_h > 0).astype(int)
    units['energy_mws'] = units.inertia_s * units.mbase
    units['gain_mw_hz'] = units.gain_pu * units.mbase / 50.0
    demand = read_inc_table(inc / 'd_seasons.inc', 2)
    wind = read_inc_table(inc / 'RES_seasons.inc', 2)
    profiles = []
    for season in ['winter','spring','summer','autumn']:
        for day in range(1,8):
            for hour in range(1,25):
                key=(f't{hour}', f'd{day}')
                profiles.append({'season':season, 'day':day, 'hour':hour,
                    'demand_mw':float(demand.loc[key,season]), 'wind_available_mw':float(wind.loc[key,season])})
    return units, pd.DataFrame(profiles), blocks

import math
from scipy.special import expit
ROOT=find_root()
UNITS,PROFILES,BLOCKS=load_inputs(ROOT)
DEP=json.loads((ROOT/'GAMS_LaPalma/LP35_Deployment.json').read_text())
EUNIT=UNITS.energy_mws.to_numpy(); GAIN=UNITS.gain_mw_hz.to_numpy(); PMAX=UNITS.pmax_mw.to_numpy()
IDS=UNITS.unit.tolist()


## 1. Recompute all 132 Big-M values from the actual fleet
The original exporter bounds the inactive-outage margin over a box. At $x_k=0$, $L=0$; therefore $c_L$ does not enter the inactive lower bound:
$$\ell_k=c_0+\min(0,c_E)\sum_{i\ne k}E_i+\min(0,c_G)\sum_{i\ne k}G_i+\min(0,c_R)\sum_{i\ne k}P_i^{\max}.$$
It exports $M_k=\max(0,-\ell_k)+10^{-5}(1+|\ell_k|)$ to twelve significant digits. This is an inactive-contingency bound, not a bound on every physically possible transient.

In [2]:
rows=[]
for name,cut in DEP['cuts'].items():
    for j in range(11):
        mask=np.arange(11)!=j
        lower=(cut['k0']+min(0,cut['kE'])*EUNIT[mask].sum()
               +min(0,cut['kG'])*GAIN[mask].sum()+min(0,cut['kR'])*PMAX[mask].sum())
        recomputed=float(f'{max(0,-lower)+1e-5*(1+abs(lower)):.12g}')
        expected=cut['bigM_by_outage'][j]
        assert recomputed==expected,(name,IDS[j],recomputed,expected)
        assert lower+expected>=0
        rows.append({'run':name,'outage':IDS[j],'big_m':expected,'absolute_error':abs(recomputed-expected)})
assert len(rows)==132
print('PASS: 132/132 actual-fleet Big-M values reproduced at the frozen serialization precision.')
display(pd.DataFrame(rows).head(11))


PASS: 132/132 actual-fleet Big-M values reproduced at the frozen serialization precision.


,run,outage,big_m,absolute_error
0,LDA,i1,294.890536,0.0
1,LDA,i2,294.890536,0.0
2,LDA,i3,294.890536,0.0
3,LDA,i4,293.736195,0.0
4,LDA,i5,286.411016,0.0
5,LDA,i6,288.193394,0.0
6,LDA,i7,276.454789,0.0
7,LDA,i8,278.502738,0.0
8,LDA,i9,278.502738,0.0
9,LDA,i10,278.502738,0.0


## 2. Reproduce the exact GAMS coefficient include
The following serialization statements follow the original `export35` function, but read **frozen manifest values**, not fitted model objects. The output remains in memory and is compared against the shipped bytes. No coefficient or repository file is replaced. Twelve significant digits and CRLF line endings are part of the authenticated export format.

In [3]:
runs=DEP['runs'];cuts=DEP['cuts'];MODEL_ID=DEP['model_id'];FMIN=DEP['nadir_min_hz']
text=[f'$setglobal LP_MODEL_ID {MODEL_ID}',f'scalar lpNadirHz /{FMIN:.12g}/;',
      'set lpRun /'+','.join(r['run'] for r in runs)+'/;',
      'parameter lpMode(lpRun),lpProbability(lpRun),lpK0(lpRun),lpKL(lpRun),lpKE(lpRun),lpKG(lpRun),lpKR(lpRun),lpBigM(lpRun,i);',
      'lpMode(lpRun)=2;lpProbability(lpRun)=0;lpBigM(lpRun,i)=0;']
for r in runs:
    n=r['run'];c=cuts[n];text.append(f"lpProbability('{n}')={r['pstar']:.12g};")
    for v,k in [('lpK0','k0'),('lpKL','kL'),('lpKE','kE'),('lpKG','kG'),('lpKR','kR')]:
        text.append(f"{v}('{n}')={c[k]:.12g};")
    for j,i in enumerate(IDS):
        text.append(f"lpBigM('{n}','{i}')={c['bigM_by_outage'][j]:.12g};")
recreated=('\r\n'.join(text)+'\r\n').encode('utf-8')
assert recreated==(ROOT/'GAMS_LaPalma/inc/LP35_Calibrated.inc').read_bytes()
assert hashlib.sha256(recreated).hexdigest()==DEP['cut_sha256']
print('PASS: frozen coefficient export is byte-for-byte identical; no fitting or file mutation.')


PASS: frozen coefficient export is byte-for-byte identical; no fitting or file mutation.


## 3. A constructed teaching operating point — not a UC result
Use the actual first winter-hour demand/wind data and turn on units i1, i5, i7 and i8. Start at their minimum outputs and distribute the remaining net load in proportion to available headroom. This is only a static balanced dispatch constructed for arithmetic teaching: it is **not** an archived or optimized schedule and need not pass every frequency screen.

Compute the post-outage aggregates independently for each online unit, then compare RoCoF, headroom and the LR_C1 affine margin. Passing a learned inequality does not by itself establish SODE safety.

In [4]:
online=np.isin(UNITS.unit,['i1','i5','i7','i8']).astype(float)
row=PROFILES.iloc[0];net_load=float(row.demand_mw-row.wind_available_mw)
power=UNITS.pmin_mw.to_numpy()*online
spare=(PMAX-UNITS.pmin_mw.to_numpy())*online
remaining=net_load-power.sum()
assert 0<=remaining<=spare.sum()
power+=remaining*spare/spare.sum()
assert abs(power.sum()+row.wind_available_mw-row.demand_mw)<1e-10
assert np.all(power>=UNITS.pmin_mw.to_numpy()*online-1e-10)
assert np.all(power<=PMAX*online+1e-10)
cut=DEP['cuts']['LR_C1'];events=[]
for j in np.flatnonzero(online):
    survive=online.copy();survive[j]=0
    E=float(EUNIT@survive);G=float(GAIN@survive)
    R=float(((PMAX*online-power)*(np.arange(11)!=j)).sum());L=float(power[j])
    margin=cut['k0']+cut['kL']*L+cut['kE']*E+cut['kG']*G+cut['kR']*R
    events.append(dict(outage=IDS[j],loss_mw=L,inertia_mws=E,gain_mw_hz=G,reserve_mw=R,
        rocof_hz_s=50*L/(2*E),headroom_margin_mw=R-L,LR_C1_margin=margin,
        affine_accept=margin>=0,fixture_only=True))
display(pd.DataFrame(events))
print('This constructed dispatch is not part of the frozen reference results.')


,outage,loss_mw,inertia_mws,gain_mw_hz,reserve_mw,rocof_hz_s,headroom_margin_mw,LR_C1_margin,affine_accept,fixture_only
0,i1,2.404105,83.8384,15.86,12.367409,0.716887,9.963304,16.442987,True,True
1,i5,3.425141,72.9696,14.26,10.508445,1.173482,7.083304,10.133192,True,True
2,i7,6.798204,60.2080,11.72,9.381508,2.822799,2.583304,-0.827457,False,True
3,i8,6.809246,62.8330,12.22,9.092550,2.709263,2.283304,-1.100107,False,True


This constructed dispatch is not part of the frozen reference results.


## 4. Why a probability threshold is still affine
For standardized features $z=(a-\mu)/\sigma$ and base score $s=w^Tz+b$, one sigmoid gives $p=\operatorname{sigmoid}(\alpha s+\beta)$. Requiring $p\ge p^*$ is equivalent to
$$c_j=\alpha w_j/\sigma_j,\qquad c_0=\alpha b+\beta-\operatorname{logit}(p^*)-\sum_j c_j\mu_j.$$
The source function below is copied without modification. It is tested with an explicitly artificial scaler/model fixture; the real fitted scaler/calibrator objects are not included and are **not** recreated by this test.

The grouped split test also uses artificial parents. All outages of a parent must remain in the same fold; row-wise splitting would leak the operating point across training and evaluation.

In [5]:
def derive_cut(artifact,pstar):
    pipe=artifact['pipeline'];cal=artifact['calibrator'];scale=pipe['scale'];m=pipe['model']
    w=np.ravel(m.coef_).astype(float);bias=float(np.ravel(m.intercept_)[0])
    c=cal['a']*w/scale.scale_;c0=cal['a']*bias+cal['b']-math.log(pstar/(1-pstar))-float(c@scale.mean_)
    # Test the actual twelve-significant-digit serialization, not just unrounded floats.
    fmt=lambda v:float(f'{v:.12g}')
    c=np.array([fmt(v) for v in c]);c0=fmt(c0)
    M=[]
    for outage in range(11):
        mask=np.arange(11)!=outage
        # Inactive candidate: its lost output is exactly zero.
        lower=c0+min(0,c[1])*EUNIT[mask].sum()+min(0,c[2])*GAIN[mask].sum()+min(0,c[3])*PMAX[mask].sum()
        M.append(fmt(max(0,-lower)+1e-5*(1+abs(lower))))
    return dict(k0=c0,kL=float(c[0]),kE=float(c[1]),kG=float(c[2]),kR=float(c[3]),bigM_by_outage=M,pstar=pstar)

def group_folds(frame,n_splits,seed):
    """Stratify PARENTS by presence of an unsafe event; all their outages stay together."""
    from sklearn.model_selection import StratifiedKFold
    groups=frame.groupby('parent_id',sort=True).y_safe.min()
    if (groups==-1).sum()<n_splits:raise ValueError('Too few distinct unsafe parent groups for the declared folds.')
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    for train_ids,test_ids in cv.split(groups.index,(groups.to_numpy()==-1).astype(int)):
        trainset=set(groups.index[train_ids]);testset=set(groups.index[test_ids])
        assert not trainset & testset
        yield frame[frame.parent_id.isin(trainset)].copy(),frame[frame.parent_id.isin(testset)].copy()


In [6]:
from types import SimpleNamespace
mean=np.array([5.,150.,20.,15.]);scale=np.array([2.,40.,5.,6.])
w=np.array([-1.1,.25,.7,1.2]);bias=.15;a=1.3;b=-.4;q=.95
artifact={'pipeline':{'scale':SimpleNamespace(mean_=mean,scale_=scale),
                     'model':SimpleNamespace(coef_=w[None,:],intercept_=np.array([bias]))},
          'calibrator':{'a':a,'b':b}}
toycut=derive_cut(artifact,q)
rng=np.random.default_rng(410);X=rng.uniform([1,20,1,1],[15,300,50,60],size=(100,4))
p_native=expit(a*(((X-mean)/scale)@w+bias)+b)
margin=toycut['k0']+X@np.array([toycut['kL'],toycut['kE'],toycut['kG'],toycut['kR']])
p_export=expit(margin+math.log(q/(1-q)))
assert np.max(np.abs(p_native-p_export))<1e-9
assert np.array_equal(p_native>=q,margin>=0)
parents=pd.DataFrame([{'parent_id':f'parent{p:02d}','y_safe':(-1 if p<9 and o==0 else 1)}
                      for p in range(18) for o in range(3)])
seen=[]
for train,test in group_folds(parents,3,20260920):
    assert not set(train.parent_id)&set(test.parent_id)
    assert all(test.groupby('parent_id').size()==3)
    seen.extend(test.parent_id.unique().tolist())
assert sorted(seen)==sorted(parents.parent_id.unique().tolist())
report={'status':'PASS','actual_big_m_values':132,'big_m_max_abs_error':0,
        'coefficient_export_byte_match':True,'toy_affine_probability_error':float(np.max(abs(p_native-p_export))),
        'toy_group_folds':3,'original_fitted_pipeline_replay':'NOT RUN'}
out=ROOT/'verification/local_reports';out.mkdir(parents=True,exist_ok=True)
(out/'01_frozen_cuts.json').write_text(json.dumps(report,indent=2)+'\n')
print('PASS: toy affine transformation and group isolation. Original fitted-pipeline replay remains NOT RUN.')


PASS: toy affine transformation and group isolation. Original fitted-pipeline replay remains NOT RUN.
